In [ ]:
# ライブラリ読み込み
import os
import numpy as np
import pandas as pd
import scipy.io
from scipy.io import mmread, mmwrite
from scipy.sparse import csr_matrix
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ann
import copy

In [ ]:
# define the variables
datadir = 'mtscATAC-seq' # please change your directory
expname = ['bulk1', 'bulk2', 'PT1_neg', 'PT1_pos', 'PT2_neg', 'PT2_pos']
matname = ['A_counts_fw', 'A_counts_rev', 'C_counts_fw', 'C_counts_rev', 'G_counts_fw', 'G_counts_rev', 'T_counts_fw', 'T_counts_rev', 'coverage']
tsvname = ['PT1_bulk1', 'PT1_bulk2', 'PT1_CD19neg', 'PT1_CD19pos', 'PT2_CD19neg', 'PT2_CD19pos']

In [ ]:
# import scATAC_data
adatas = []

for x in range(len(expname)):
    
    genes = pd.read_csv(os.path.join(datadir,'rds_output', expname[x],'genes.tsv'), sep="\t")
    genes = genes.drop(columns='strand')

    barcode = pd.read_csv(os.path.join(datadir,'rds_output', expname[x],'barcodes.tsv'), sep='\t', names=['barcode'])
    celltype = pd.read_csv(os.path.join('mtscATAC-seq/mtscATACpaper_reproducibility-master/figure_CLL/output/projection_dfs/CLL_'+ tsvname[x]+'_projection.tsv'),sep='\t')
    barcode = pd.merge(barcode, celltype,  how="inner", on = "barcode")
    
    mtx = scipy.io.mmread(os.path.join(datadir,'rds_output',expname[x],'matrix.mtx'))
    mtx = mtx.toarray()
    mtx = mtx.T
    
    adata = ann.AnnData(X=mtx, obs=barcode, var=genes)
    adata.var_names_make_unique()
    adatas.append(adata)

In [ ]:
# import mtscATAC_data
adatas_mg = []

for x in range(len(expname)):
    
    del_index = pd.read_csv(os.path.join(datadir,'row_del/CLL_' + tsvname[x] + '_del.tsv'),sep='\t')
    del_index = np.array(del_index['index'])
    
    genes = pd.read_csv(os.path.join(datadir,'rds_output', expname[x] + '_mg', 'genes.tsv'), sep='\t')
    genes = genes.drop(columns=['strand','width'])
    
    barcode = pd.read_csv(os.path.join(datadir,'rds_output', expname[x] + '_mg', 'barcodes.tsv'), sep="\t", names=['barcode'])
    barcode = pd.DataFrame(barcode['barcode'][del_index])
    
    mtx = scipy.io.mmread(os.path.join(datadir,'rds_output', expname[x] + '_mg', 'coverage.mtx'))
    mtx = mtx.toarray()
    mtx = mtx.T[del_index,:]
    
    adata_mg = ann.AnnData(X=mtx, obs=barcode, var=genes)
    adata_mg.obs.reset_index(inplace=True, drop=True)
    
    for y in range(len(matname)):
        mtx = scipy.io.mmread(os.path.join(datadir,'rds_output', expname[x] + '_mg', matname[y]+'.mtx'))
        mtx = mtx.toarray()
        mtx = mtx.T[del_index,:]
        adata_mg.layers[matname[y]] = mtx
        
    celltype = pd.read_csv(os.path.join(datadir,'mtscATACpaper_reproducibility-master/figure_CLL/output/projection_dfs/CLL_'+ tsvname[x]+'_projection.tsv'),sep='\t')
    adata_mg.obs = pd.merge(adata_mg.obs, celltype,  how="inner", on = "barcode")
    
    adata_mg.var_names_make_unique() # this is unnecessary if using `var_names='gene_ids'` in `sc.read_10x_mtx`
    adatas_mg.append(adata_mg)

In [ ]:
# import mtscATAC_data

del_index = pd.read_csv(os.path.join(datadir,'row_del/CLL_' + tsvname[5] + '_del.tsv'),sep='\t')
del_index = np.array(del_index['index'])
    
genes = pd.read_csv(os.path.join(datadir,'rds_output', expname[5] + '_mg', 'genes.tsv'), sep='\t')
genes = genes.drop(columns=['strand','width'])
    
barcode = pd.read_csv(os.path.join(datadir,'rds_output', expname[5] + '_mg', 'barcodes.tsv'), sep="\t", names=['barcode'])
barcode = pd.DataFrame(barcode['barcode'][del_index])
    
mtx = scipy.io.mmread(os.path.join(datadir,'rds_output', expname[5] + '_mg', 'coverage.mtx'))
mtx = mtx.toarray()
mtx = mtx.T[del_index,:]
    
adata_mg = ann.AnnData(X=mtx, obs=barcode, var=genes)
adata_mg.obs.reset_index(inplace=True, drop=True)
    
for y in range(len(matname)):
    mtx = scipy.io.mmread(os.path.join(datadir,'rds_output', expname[5] + '_mg', matname[y]+'.mtx'))
    mtx = mtx.toarray()
    mtx = mtx.T[del_index,:]
    adata_mg.layers[matname[y]] = mtx
        
celltype = pd.read_csv(os.path.join(datadir,'mtscATACpaper_reproducibility-master/figure_CLL/output/projection_dfs/CLL_'+ tsvname[5]+'_projection.tsv'),sep='\t')
adata_mg.obs = pd.merge(adata_mg.obs, celltype,  how="inner", on = "barcode")
    
adata_mg.var_names_make_unique() # this is unnecessary if using `var_names='gene_ids'` in `sc.read_10x_mtx`
adatas_mg.append(adata_mg)

In [ ]:
adatas_mg

In [ ]:
f = []
for i in range(len(expname)):
    celltype = pd.read_csv(os.path.join(datadir,'mtscATACpaper_reproducibility-master/figure_CLL/output/projection_dfs/CLL_'+ tsvname[i]+'_projection.tsv'),sep='\t')
    f.append(celltype)
    print(len(f[i]))

In [ ]:
len(celltype)

In [ ]:
# ref Alleleの行番号以外の行番号を抽出

not_A_index = np.array(adatas_mg[0].var.reset_index().query('refAllele!="A"').index)
not_C_index = np.array(adatas_mg[0].var.reset_index().query('refAllele!="C"').index)
not_G_index = np.array(adatas_mg[0].var.reset_index().query('refAllele!="G"').index)
not_T_index = np.array(adatas_mg[0].var.reset_index().query('refAllele!="T"').index)

In [ ]:
adatas_mg_VAF = copy.deepcopy(adatas_mg)


for x in range(len(expname)):
    del adatas_mg_VAF[x].layers
    
    #adatas_mg_VAF[x].layers['variant_refA'] = adatas_mg[x].X - adatas_mg[x].layers['C_counts_fw'] - adatas_mg[x].layers['C_counts_rev'] - adatas_mg[x].layers['G_counts_fw'] - adatas_mg[x].layers['G_counts_rev'] - adatas_mg[x].layers['T_counts_fw'] - adatas_mg[x].layers['T_counts_rev']
    adatas_mg_VAF[x].layers['variant_refA'] = adatas_mg[x].X - adatas_mg[x].layers['A_counts_fw'] - adatas_mg[x].layers['A_counts_rev']
    adatas_mg_VAF[x].layers['variant_refA'][:,not_A_index] = 0
    
    #adatas_mg_VAF[x].layers['variant_refC'] = adatas_mg[x].X - adatas_mg[x].layers['A_counts_fw'] - adatas_mg[x].layers['A_counts_rev'] - adatas_mg[x].layers['G_counts_fw'] - adatas_mg[x].layers['G_counts_rev'] - adatas_mg[x].layers['T_counts_fw'] - adatas_mg[x].layers['T_counts_rev']
    adatas_mg_VAF[x].layers['variant_refC'] = adatas_mg[x].X - adatas_mg[x].layers['C_counts_fw'] - adatas_mg[x].layers['C_counts_rev']
    adatas_mg_VAF[x].layers['variant_refC'][:,not_C_index] = 0
    
    #adatas_mg_VAF[x].layers['variant_refG'] = adatas_mg[x].X - adatas_mg[x].layers['A_counts_fw'] - adatas_mg[x].layers['A_counts_rev'] - adatas_mg[x].layers['C_counts_fw'] - adatas_mg[x].layers['C_counts_rev'] - adatas_mg[x].layers['G_counts_fw'] - adatas_mg[x].layers['G_counts_rev']
    adatas_mg_VAF[x].layers['variant_refG'] = adatas_mg[x].X - adatas_mg[x].layers['G_counts_fw'] - adatas_mg[x].layers['G_counts_rev']
    adatas_mg_VAF[x].layers['variant_refG'][:,not_G_index] = 0
    
    #adatas_mg_VAF[x].layers['variant_refT'] = adatas_mg[x].X - adatas_mg[x].layers['A_counts_fw'] - adatas_mg[x].layers['A_counts_rev'] - adatas_mg[x].layers['C_counts_fw'] - adatas_mg[x].layers['C_counts_rev'] - adatas_mg[x].layers['G_counts_fw'] - adatas_mg[x].layers['G_counts_rev']
    adatas_mg_VAF[x].layers['variant_refT'] = adatas_mg[x].X - adatas_mg[x].layers['T_counts_fw'] - adatas_mg[x].layers['T_counts_rev']
    adatas_mg_VAF[x].layers['variant_refT'][:,not_T_index] = 0
    
    adatas_mg_VAF[x].layers['all_mut'] = adatas_mg_VAF[x].layers['variant_refA'] + adatas_mg_VAF[x].layers['variant_refC'] + adatas_mg_VAF[x].layers['variant_refG'] + adatas_mg_VAF[x].layers['variant_refT']
    adatas_mg_VAF[x].layers['VAF'] = adatas_mg_VAF[x].layers['all_mut'] / adatas_mg_VAF[x].X
    adatas_mg_VAF[x].layers['VAF'] = np.nan_to_num(adatas_mg_VAF[x].layers['VAF'])
    adatas_mg_VAF[x].obs['VAF'] = adatas_mg_VAF[x].layers['all_mut'].sum(axis=1) / adatas_mg_VAF[x].X.sum(axis=1)

In [ ]:
adatas_mg_VAF[0].obs

In [ ]:
for i in range(len(expname)):
    print(max(adatas_mg_VAF[i].obs['VAF']))

In [ ]:
vaf_vec = np.array(adatas_mg_VAF[2].layers['all_mut'].sum(axis=0)/adatas_mg_VAF[2].X.sum(axis=0)).reshape(-1)

In [ ]:
len(vaf_vec>0.01)

In [ ]:
plt.hist(vaf_vec , bins=100)

In [ ]:
print((vaf_vec > 0.01 ).sum())

In [ ]:
plt.hist(np.log10(vaf_vec[vaf_vec > 0.01]), bins=100)

In [ ]:
np.log10(vaf_vec[np.logical_and(np.log10(vaf_vec) > -1.25, np.log10(vaf_vec) < -1.1)][0])

In [ ]:
True * False

In [ ]:
#fig,ax = plt.subplots(nrows=1,ncols=6)
#ax = ax.flatten()

for i in range(len(adatas_mg_VAF)):
    #sns.heatmap(adatas_mg_VAF[i].layers['all_mut'], ax=ax[i])
    print(max(adatas_mg_VAF[i].layers['all_mut'][0]))
    print(max(adatas_mg_VAF[i].X[0]))

In [ ]:
for i in range(len(expname)):
    adatas_mg_VAF[i].layers['all_mut'].sum(axis=0) < 1

In [ ]:
adatas_mg_VAF[i].layers['all_mut']

In [ ]:
layername = ['variant_refA', 'variant_refC', 'variant_refG', 'variant_refT', 'all_mut', 'VAF']

for x in range(len(expname)):
    for y in range(len(layername)-2):
        del adatas_mg_VAF[x].layers[layername[y]]

In [ ]:
layername = ['all_mut', 'VAF']

for x in range(len(expname)):
    adatas[x].X = csr_matrix(adatas[x].X)
    for y in range(len(layername)):
        adatas_mg_VAF[x].layers[layername[y]] = csr_matrix(adatas_mg_VAF[x].layers[layername[y]])

In [ ]:
dirname_h5ad = os.path.join(datadir,'h5ad_files')
if not os.path.exists(dirname_h5ad):
    os.makedirs(dirname_h5ad)
    
for x in range(len(expname)):
    adatas[x].write(os.path.join(dirname_h5ad, expname[x] + '.h5ad'), compression='gzip')
    adatas_mg_VAF[x].write(os.path.join(dirname_h5ad, expname[x] + '_mg.h5ad'), compression='gzip')